In [1]:
import pandas as pd

In [2]:
from pathlib import Path
base = Path("../../tasks/task_01/data/")

In [3]:
customer_df = pd.read_csv(base/ "customers.csv")
product_df = pd.read_csv(base/ "products.csv")
order_df = pd.read_csv(base/ "orders.csv")
returns_df = pd.read_csv(base/ "returns.csv")

Q1:What is the total net revenue for customers in the west region after subtracting returned units at the original discounted unit price, rounded to 2 decimals?

In [4]:
orders = order_df.merge(customer_df, on="customer_id").merge(product_df, on="product_id")
returned = returns_df.groupby("order_id", as_index=False)["returned_qty"].sum()
orders = orders.merge(returned, on="order_id", how="left").fillna({"returned_qty": 0})
orders["net_revenue"] = (orders["quantity"] - orders["returned_qty"]) * orders["unit_price"] * (1 - orders["discount_pct"] / 100)
west_net_revenue = orders.loc[orders['region'] == 'West', 'net_revenue'].sum()
print(f"{west_net_revenue:.2f}")

1521.95


Q2: Returned units reduce revenue on the day the return is processed. On which date did total net revenue have the largest increase versus the previous date?

In [5]:
orders_for_returns = order_df.merge(product_df, on="product_id")
orders_for_returns["gross_revenue"] = orders_for_returns["quantity"] * orders_for_returns["unit_price"] * (1 - orders_for_returns["discount_pct"] / 100)
daily_gross = orders_for_returns.groupby("order_date", as_index=False)["gross_revenue"].sum()

returns_with_prices = returns_df.merge(orders_for_returns[["order_id", "unit_price", "discount_pct"]], on="order_id", how="left")
returns_with_prices["return_chargeback"] = returns_with_prices["returned_qty"] * returns_with_prices["unit_price"] * (1 - returns_with_prices["discount_pct"] / 100)
daily_returns = returns_with_prices.groupby("return_date", as_index=False)["return_chargeback"].sum()

daily = daily_gross.merge(daily_returns, left_on="order_date", right_on="return_date", how="outer")
daily["date"] = daily["order_date"].fillna(daily["return_date"])
daily = daily[["date", "gross_revenue", "return_chargeback"]].fillna(0).sort_values("date")
daily["net_revenue"] = daily["gross_revenue"] - daily["return_chargeback"]
daily["change"] = daily["net_revenue"].diff()
max_change_date = pd.Timestamp(daily.loc[daily["change"].idxmax(), "date"]).date().isoformat()
print(max_change_date)

2024-01-15


Q3: Returned units are charged back to the original sale date. On which date did total net revenue have the largest increase versus the previous date?

In [6]:
daily = orders.groupby("order_date", as_index=False)["net_revenue"].sum().sort_values("order_date")
daily["change"] = daily["net_revenue"].diff()
max_change_date = pd.Timestamp(daily.loc[daily["change"].idxmax(), "order_date"]).date().isoformat()
print(max_change_date)

2024-01-27


Q4: Using order_date as the date of record, among dates with at least 500 of gross revenue, on which date did returned units consume the largest share of that day's gross revenue?

In [7]:
orders = pd.read_csv(base / "orders.csv")
returns_df = pd.read_csv(base / "returns.csv")
products = pd.read_csv(base / "products.csv")

orders = orders.merge(products, on="product_id")
returned = returns_df.groupby("order_id", as_index=False)["returned_qty"].sum()
orders = orders.merge(returned, on="order_id", how="left").fillna({"returned_qty": 0})

orders["gross_revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount_pct"] / 100)
orders["return_chargeback"] = orders["returned_qty"] * orders["unit_price"] * (1 - orders["discount_pct"] / 100)

daily = orders.groupby("order_date", as_index=False)[["gross_revenue", "return_chargeback"]].sum()
daily = daily[daily["gross_revenue"] >= 500].copy()
daily["return_share"] = daily["return_chargeback"] / daily["gross_revenue"]

q4 = str(daily.sort_values(["return_share", "order_date"], ascending=[False, True]).iloc[0]["order_date"])
print(q4)

2024-01-05
